In [13]:
import torch

x = torch.tensor([[ 1.0, -2.0],
                      [-1.0,  1.0]])
linear = torch.nn.Linear(2, 3, bias=False)
weight = torch.tensor([[ 0.2, -0.1],
                       [-0.3,  0.1],
                       [ 0.2,  0.2]])
linear.weight.data = x
y_linear = linear(x)
y_matmul = x.matmul(weight.T)
y_relu = y_matmul.relu()
print(f"y_linear:\n{y_linear}")
print(f"y_matmul:\n{y_matmul}")
print(f"y_relu:\n{y_relu}")

y_linear:
tensor([[ 5., -3.],
        [-3.,  2.]], grad_fn=<MmBackward0>)
y_matmul:
tensor([[ 0.4000, -0.5000, -0.2000],
        [-0.3000,  0.4000,  0.0000]])
y_relu:
tensor([[0.4000, 0.0000, 0.0000],
        [0.0000, 0.4000, 0.0000]])


# BitNet: Scaling 1-bit Transformers for Large Language Models

In [243]:
import torch

x = torch.tensor([[ 1.0, -2.0],
                  [-1.0,  1.0]])
w = torch.tensor([[ 0.2, -0.1],
                  [-0.3,  0.1],
                  [ 0.2,  0.2]])
alpha = w.mean()
tilde_w = torch.sign(w - alpha)
b = 8
q_b = 2**(b-1)
gamma = w.abs().max()
epsilon = 1e-5
def quant(x: torch.Tensor) -> torch.Tensor:
  eta = x.min()
  return torch.clip((x - eta) * q_b / gamma, epsilon, q_b - epsilon)
layer_norm = torch.nn.LayerNorm(normalized_shape=2, elementwise_affine=False, bias=False)
beta = w.abs().mean()
tilde_x = quant(layer_norm(x)) * ( (beta * gamma) / q_b )
y = tilde_x.matmul(tilde_w.T)
print(f"tilde_w:\n{tilde_w}")
print(f"tilde_x:\n{tilde_x}")
print(f"y:\n{y}")

tilde_w:
tensor([[ 1., -1.],
        [-1.,  1.],
        [ 1.,  1.]])
tilde_x:
tensor([[5.5000e-02, 4.2969e-09],
        [5.1359e-07, 5.5000e-02]])
y:
tensor([[ 0.0550, -0.0550,  0.0550],
        [-0.0550,  0.0550,  0.0550]])


In [244]:
import sys
import os
src_path = os.path.abspath(os.path.join('..'))
if src_path not in sys.path:
  sys.path.append(src_path)

from importlib import reload
from src import bitlinear
reload(bitlinear)
from src.bitlinear import BitLinear

x = torch.tensor([[ 1.0, -2.0],
                  [-1.0,  1.0]])
w = torch.tensor([[ 0.2, -0.1],
                  [-0.3,  0.1],
                  [ 0.2,  0.2]])

bitlinear = BitLinear(2, 3, b=2)
bitlinear.weight.data = w
y = bitlinear(x)
print(f"y:\n{y}")

y:
tensor([[ 0.3667, -0.3667,  0.3667],
        [-0.3667,  0.3667,  0.3667]], grad_fn=<MmBackward0>)


In [249]:
x = torch.arange(3 * 2 * 4).reshape(3, 2, 4).float()
w = torch.arange(3 * 4).reshape(3, 4).float()
bitlinear = BitLinear(4, 3, b=8)
bitlinear.weight.data = w
y = bitlinear(x)
print(f"x:\n{x}")
print(f"w:\n{w}")
print(f"y:\n{y}")

x:
tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.]],

        [[ 8.,  9., 10., 11.],
         [12., 13., 14., 15.]],

        [[16., 17., 18., 19.],
         [20., 21., 22., 23.]]])
w:
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
y:
tensor([[[-29.5160,  19.6773,  29.5160],
         [-29.5160,  19.6773,  29.5160]],

        [[-29.5160,  19.6773,  29.5160],
         [-29.5160,  19.6773,  29.5160]],

        [[-29.5160,  19.6773,  29.5160],
         [-29.5160,  19.6773,  29.5160]]], grad_fn=<UnsafeViewBackward0>)


# The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits

In [270]:
import torch

x = torch.tensor([[ 1.0, -2.0],
                  [-1.0,  1.0]])
w = torch.tensor([[ 0.2, -0.1],
                  [-0.3,  0.1],
                  [ 0.2,  0.2]])
w_gamma = w.abs().mean()
tilde_w = torch.clip(torch.round(w / (w_gamma + epsilon)), -1, 1)
b = 2
q_b = 8**(b-1)
epsilon = 1e-5
x_gamma = x.abs().mean()
def quant(x: torch.Tensor) -> torch.Tensor:
  return torch.clip(x * q_b / x_gamma, -q_b + epsilon, q_b - epsilon)
layer_norm = torch.nn.LayerNorm(normalized_shape=2, elementwise_affine=False, bias=False)
beta = w.abs().mean()
tilde_x = quant(layer_norm(x)) * ( (beta * x_gamma) / q_b )
y = tilde_x.matmul(tilde_w.T)
print(f"tilde_w:\n{tilde_w}")
print(f"tilde_x:\n{tilde_x}")
print(f"y:\n{y}")

tilde_w:
tensor([[ 1., -1.],
        [-1.,  1.],
        [ 1.,  1.]])
tilde_x:
tensor([[ 0.1833, -0.1833],
        [-0.1833,  0.1833]])
y:
tensor([[ 0.3667, -0.3667,  0.0000],
        [-0.3667,  0.3667,  0.0000]])


In [278]:
import sys
import os
src_path = os.path.abspath(os.path.join('..'))
if src_path not in sys.path:
  sys.path.append(src_path)

from importlib import reload
from src import bitlinear_b158
reload(bitlinear_b158)
from src.bitlinear_b158 import BitLinearb158

x = torch.tensor([[ 1.0, -2.0],
                  [-1.0,  1.0]])
w = torch.tensor([[ 0.2, -0.1],
                  [-0.3,  0.1],
                  [ 0.2,  0.2]])

bitlinear = BitLinearb158(2, 3, b=2)
bitlinear.weight.data = w
y = bitlinear(x)
print(f"y:\n{y}")

y:
tensor([[ 0.3667, -0.3667,  0.0000],
        [-0.3667,  0.3667,  0.0000]], grad_fn=<MmBackward0>)
